In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import time
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
from sklearn.metrics import f1_score

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

In [ ]:

start = time.time()

grid_search = GridSearchCV(rf, param_grid, cv=5, scoring="f1", n_jobs=-1)

grid_search.fit(X_train, y_train)

grid_time = time.time() - start

In [ ]:
print("Best Parameters:", grid_search.best_params_)
print("Best F1 Score:", grid_search.best_score_)
print("Grid Search Time:", grid_time)

Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Best F1 Score: 0.9684436877471334
Grid Search Time: 38.06276297569275


In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_)

grid_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]]

,params,mean_test_score,std_test_score,rank_test_score
0,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.961730,0.019980,24
1,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.963408,0.018366,12
2,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.968444,0.015105,1
3,"{'max_depth': None, 'min_samples_leaf': 2, 'n_...",0.966856,0.022092,3
4,"{'max_depth': None, 'min_samples_leaf': 2, 'n_...",0.965090,0.019728,8
5,"{'max_depth': None, 'min_samples_leaf': 2, 'n_...",0.963382,0.018481,14
6,"{'max_depth': None, 'min_samples_leaf': 4, 'n_...",0.961761,0.020026,20
7,"{'max_depth': None, 'min_samples_leaf': 4, 'n_...",0.966828,0.019962,5
8,"{'max_depth': None, 'min_samples_leaf': 4, 'n_...",0.963382,0.018481,14
9,"{'max_depth': 5, 'min_samples_leaf': 1, 'n_est...",0.961855,0.019998,19


In [ ]:
param_dist = {
    "n_estimators": randint(50, 501),
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_leaf": randint(1, 11)
}

In [ ]:
start = time.time()

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

random_time = time.time() - start

In [ ]:
print("Best Parameters:", random_search.best_params_)
print("Best F1 Score:", random_search.best_score_)
print("Random Search Time:", random_time)


Best Parameters: {'max_depth': 30, 'min_samples_leaf': 2, 'n_estimators': 241}
Best F1 Score: 0.9667669294797836
Random Search Time: 74.08253455162048


In [ ]:
random_results = pd.DataFrame(random_search.cv_results_)

random_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]]

,params,mean_test_score,std_test_score,rank_test_score
0,"{'max_depth': 15, 'min_samples_leaf': 8, 'n_es...",0.959991,0.017578,19
1,"{'max_depth': 20, 'min_samples_leaf': 7, 'n_es...",0.959991,0.017578,19
2,"{'max_depth': 10, 'min_samples_leaf': 7, 'n_es...",0.961729,0.018330,12
3,"{'max_depth': 10, 'min_samples_leaf': 8, 'n_es...",0.961729,0.018330,12
4,"{'max_depth': 15, 'min_samples_leaf': 8, 'n_es...",0.961700,0.019190,15
5,"{'max_depth': 10, 'min_samples_leaf': 6, 'n_es...",0.961761,0.020026,11
6,"{'max_depth': 5, 'min_samples_leaf': 8, 'n_est...",0.963437,0.019731,6
7,"{'max_depth': 30, 'min_samples_leaf': 2, 'n_es...",0.966767,0.016792,1
8,"{'max_depth': 15, 'min_samples_leaf': 5, 'n_es...",0.963347,0.017456,8
9,"{'max_depth': 15, 'min_samples_leaf': 10, 'n_e...",0.963347,0.017456,8


In [ ]:
best_model = grid_search.best_estimator_

test_pred = best_model.predict(X_test)

print("Test F1 Score:", f1_score(y_test, test_pred))

Test F1 Score: 0.9655172413793104


## Why should we not tune on test data?

1. Test data should not be used for hyperparameter tuning because it can cause data leakage.

2. Repeated tuning on test data may lead to overfitting.

3. We should keep test data separate for unbiased final model evaluation.